# 01 — Dataset Selection & Exploratory Data Analysis (EDA)
## Maharah GDP | Call Center Agent Performance Analysis

**Objective:** validate the chosen public dataset against its official documentation, profile the entire *metadata* collection, select a small representative set of calls, inspect one original audio file and save a reproducible sample manifest. **No ASR is run in this notebook.**

**Dataset:** [AppTek Call-Center Dialogues](https://huggingface.co/datasets/apptek-com/apptek_callcenter_dialogues) — `diarization` configuration. Its **mixed audio** is the actual input for later transcription + diarization; time-stamped text and speaker roles are **reference labels for evaluation**, not model predictions.

**Research survey:** 16 datasets, 15 papers. AppTek 96/100; SpokenWOZ 85/100 (weighted research criteria; *not* a model performance metric).

**Responsible use:** public, role-played conversations; CC BY-SA 4.0. Official card says for evaluation/analysis, not model training. Avoid committing raw audio to GitHub. Do not describe reference speaker labels as the output of your diarization model.

**Colab setup:** Runtime → Change runtime type → Python 3 (CPU is enough for this notebook). Run all cells in order. Internet access is required in Colab to obtain the dataset metadata and the one sample WAV.

### Expected outputs
- `data/processed/dataset_profile.json`: quantitative profiling and integrity checks.
- `data/processed/selected_calls.csv`: *12-call stratified shortlist* (4 accents × 3 varied domains, deterministic seed).
- `data/processed/selected_calls.json`: metadata for downstream notebook use.
- `data/raw_samples/`: initially **one** audio WAV, for a physical download and playback smoke test.
- `data/processed/provenance.json`: dataset repository and exact revision SHA for reproducibility.

For final processing, this shortlist can be expanded to 20 calls after we verify transcription and diarization quality on 1–2 examples.

In [ ]:
%pip -q install "huggingface_hub>=0.26,<2" "pandas>=2.0" "matplotlib>=3.7" "soundfile>=0.12" "tqdm>=4.60"

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json, random, posixpath
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

REPO_ID = "apptek-com/apptek_callcenter_dialogues"
CONFIG = "diarization"
SEED = 42
ACCENTS_TARGET = ["en-US_General", "en-GB", "en-IN", "en-AU"]
CALLS_PER_ACCENT = 3
ROOT = Path("data")
META_DIR = ROOT / "metadata"
RAW_DIR = ROOT / "raw_samples"
OUT_DIR = ROOT / "processed"
for folder in [META_DIR, RAW_DIR, OUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print("Working directory:", Path.cwd().resolve())

## 1. Pin the dataset version and discover the *metadata-only* files

**Important:** do **not** download the whole dataset. We first download small `metadata.jsonl` files; one selected audio file is downloaded in Step 5. A normal `load_dataset(..., split="test")` may trigger unnecessary large downloads for our sample-based workflow.

In [ ]:
api = HfApi()
info = api.dataset_info(REPO_ID)
revision = info.sha
files = api.list_repo_files(REPO_ID, repo_type="dataset", revision=revision)
metadata_paths = sorted(
    p for p in files if p.startswith(f"{CONFIG}/") and p.endswith("/metadata.jsonl")
)
print(f"Dataset revision: {revision}")
print(f"Available metadata groups: {len(metadata_paths)}")
print("Examples:", metadata_paths[:5])
assert metadata_paths, "No diarization metadata found; check the repository structure."
(OUT_DIR / "provenance.json").write_text(json.dumps({
    "repo_id": REPO_ID,
    "config": CONFIG,
    "revision": revision,
    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    "license": "CC BY-SA 4.0",
    "use": "Evaluation and analysis only; no ASR training/fine-tuning",
    "metadata_paths": metadata_paths,
}, indent=2, ensure_ascii=False), encoding="utf-8")

## 2. Load metadata from all accents (no WAV files)

Every row in the `diarization` configuration represents a merged, two-speaker conversation. The `segments` list contains reference start/end timestamps, role (`customer` / `agent`), speaker ID and text.

In [ ]:
calls = []
for metadata_path in tqdm(metadata_paths, desc="Metadata groups"):
    local = hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset", filename=metadata_path,
        revision=revision, local_dir=str(META_DIR)
    )
    with open(local, encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            row["_metadata_path"] = metadata_path
            row["_line_number"] = line_no
            calls.append(row)
print("Calls loaded:", len(calls))
print("First-row fields:", list(calls[0].keys()))
print("First-segment fields:", list(calls[0].get("segments", [{}])[0].keys()))
assert calls, "No metadata rows found."

## 3. Validate fields, roles, timestamps, durations and basic coverage

The checks are diagnostics rather than automatic claims of dataset correctness. Natural overlaps between speakers are expected, so we do **not** mark overlapping segments as invalid by themselves.

In [ ]:
records = []
violations = []
for i, row in enumerate(calls):
    segs = row.get("segments") or []
    roles = {str(s.get("role", "")).lower() for s in segs}
    duration = row.get("duration")
    row_id = f"row_{i:04d}"
    filename = row.get("file_name", "")
    if not filename:
        violations.append((row_id, "missing file_name"))
    if duration is None or float(duration) <= 0:
        violations.append((row_id, "missing/invalid duration"))
    if not {"agent", "customer"}.issubset(roles):
        violations.append((row_id, f"both roles not found: {sorted(roles)}"))
    invalid_times = 0
    missing_text = 0
    for s in segs:
        start, end = s.get("start"), s.get("end")
        if start is None or end is None or float(start) < 0 or float(end) <= float(start) or (duration and float(end) > float(duration) + 1.0):
            invalid_times += 1
        if not str(s.get("text", "")).strip():
            missing_text += 1
    if invalid_times:
        violations.append((row_id, f"invalid timestamps in {invalid_times} segments"))
    records.append({
        "row_id": row_id, "file_name": filename, "accent": row.get("accent"),
        "domain": row.get("domain"), "duration_sec": float(duration) if duration else None,
        "segments": len(segs), "customer_segments": sum(str(s.get("role", "")).lower() == "customer" for s in segs),
        "agent_segments": sum(str(s.get("role", "")).lower() == "agent" for s in segs),
        "missing_text_segments": missing_text, "invalid_timestamp_segments": invalid_times,
        "metadata_path": row["_metadata_path"],
    })
df = pd.DataFrame(records)
print(f"Calls: {len(df)} | accents: {df.accent.nunique()} | domains: {df.domain.nunique()}")
print(f"Total call duration: {df.duration_sec.sum()/3600:.1f} hours")
print(f"Total reference segments: {df.segments.sum():,}")
print(f"Missing transcript segments: {df.missing_text_segments.sum():,}")
print(f"Timestamp violations: {df.invalid_timestamp_segments.sum():,}")
print(f"Calls with issue(s): {len(set(r for r, _ in violations))}")
display(df.head(6))
if violations:
    display(pd.DataFrame(violations[:30], columns=["row_id", "issue"]))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
df.accent.value_counts().sort_values().plot(kind="barh", ax=ax)
ax.set(title="Number of Calls by Accent", xlabel="Calls", ylabel="Accent")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
df.domain.value_counts().sort_values().plot(kind="barh", ax=ax)
ax.set(title="Number of Calls by Service Domain", xlabel="Calls", ylabel="Domain")
plt.tight_layout()
plt.show()

display(df[["duration_sec", "segments", "customer_segments", "agent_segments"]].describe().round(2))

## 4. Reproducible representative sampling (12 calls)

We select **3 different domains per accent**, across 4 requested accents, using a fixed seed. This is a *convenience-stratified sample for a practical assessment*, not a statistically representative test set of all accents or calls. We also retain metadata pointers for traceability.

**Call count is a starting point.** We will consider extending to ~20 only after the end-to-end pipeline runs and we estimate compute time.

In [ ]:
rng = random.Random(SEED)
selected = []
for accent in ACCENTS_TARGET:
    group = df[df.accent == accent]
    if group.empty:
        raise ValueError(f"Accent {accent} not present. Available accents: {sorted(df.accent.dropna().unique())}")
    domains = sorted(group.domain.dropna().unique())
    if len(domains) < CALLS_PER_ACCENT:
        raise ValueError(f"Only {len(domains)} unique domains for accent {accent}")
    rng.shuffle(domains)
    for domain in domains[:CALLS_PER_ACCENT]:
        candidates = group[group.domain == domain]
        # Use stable row ordering and seeded RNG for reproducibility
        chosen_idx = rng.randrange(len(candidates))
        selected.append(candidates.sort_values("file_name").iloc[chosen_idx].to_dict())
sample = pd.DataFrame(selected).reset_index(drop=True)
sample.insert(0, "call_id", [f"CALL_{i:03d}" for i in range(1, len(sample)+1)])
cols = ["call_id", "accent", "domain", "duration_sec", "segments", "customer_segments", "agent_segments", "file_name", "metadata_path", "row_id"]
sample[cols].to_csv(OUT_DIR / "selected_calls.csv", index=False)
sample[cols].to_json(OUT_DIR / "selected_calls.json", orient="records", indent=2, force_ascii=False)
print("Shortlist saved. Count:", len(sample))
display(sample[["call_id", "accent", "domain", "duration_sec", "segments"]])

## 5. Download **one** merged two-speaker audio WAV and listen

This step is the practical check that we can obtain the original **audio bytes**, not only metadata or transcripts. It downloads **one** WAV file, checks its sample rate/channels/duration, and plays the first 30 seconds. Later we will download only the full shortlist.

The audio relative path in `file_name` is joined with the directory containing its metadata file.

In [ ]:
first = sample.iloc[0]
repo_audio_path = posixpath.normpath(posixpath.join(
    posixpath.dirname(first["metadata_path"]), first["file_name"]
))
assert repo_audio_path in set(files), (
    f"Audio path not found: {repo_audio_path}. Inspect metadata field and repo tree."
)
audio_local = hf_hub_download(
    repo_id=REPO_ID, repo_type="dataset", filename=repo_audio_path,
    revision=revision, local_dir=str(RAW_DIR)
)
audio_info = sf.info(audio_local)
print("Call:", first["call_id"], "| accent:", first["accent"], "| domain:", first["domain"])
print("Repo path:", repo_audio_path)
print("Audio info:", audio_info)
print("Metadata duration(s):", round(first["duration_sec"], 2))
assert audio_info.frames > 0, "Downloaded audio file is empty."
assert audio_info.channels == 1, "Expected mono mixed audio; inspect if dataset structure changed."
assert abs(audio_info.duration - first["duration_sec"]) < 2.0, "Unexpected duration mismatch: inspect."
# Load only the first 30 seconds, not the entire recording.
with sf.SoundFile(audio_local, "r") as f:
    head = f.read(frames=min(f.frames, f.samplerate*30), dtype="float32")
    sample_rate = f.samplerate
display(Audio(head, rate=sample_rate))

## 6. Inspect the reference speaker turns (never confuse these with predictions)

The following roles, timestamps and texts are *gold/reference annotations* supplied by AppTek. In the next notebooks, we will process the audio independently, compare inferred turns/words to these references, and report errors honestly.

In [ ]:
chosen = calls[int(first["row_id"].split("_")[1])]
turns = pd.DataFrame(chosen["segments"])
print("Reference segments:", len(turns))
turn_columns = [c for c in ["start", "end", "role", "speaker_id", "text"] if c in turns.columns]
display(turns[turn_columns].head(15))
print("First 10 reference speaker roles:", turns["role"].head(10).tolist())

## 7. Save a machine-readable EDA report (and perform final checks)

In [ ]:
report = {
    "dataset": REPO_ID,
    "config": CONFIG,
    "revision": revision,
    "total_calls_in_config": int(len(df)),
    "accent_count": int(df.accent.nunique()),
    "domain_count": int(df.domain.nunique()),
    "total_call_audio_hours": round(float(df.duration_sec.sum()/3600), 2),
    "total_reference_segments": int(df.segments.sum()),
    "calls_with_both_roles": int(((df.agent_segments > 0) & (df.customer_segments > 0)).sum()),
    "total_missing_text_segments": int(df.missing_text_segments.sum()),
    "total_invalid_timestamp_segments": int(df.invalid_timestamp_segments.sum()),
    "sample_size": int(len(sample)),
    "sample_accents": sorted(sample.accent.unique().tolist()),
    "sample_domains": sorted(sample.domain.unique().tolist()),
    "random_seed": SEED,
    "first_audio_repo_path": repo_audio_path,
    "first_audio_sample_rate": audio_info.samplerate,
    "first_audio_channels": audio_info.channels,
    "first_audio_duration_seconds": round(audio_info.duration, 3),
    "note": "Reference segments/roles are dataset annotations, not model outputs; this notebook does not perform ASR.",
}
(OUT_DIR / "dataset_profile.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))
assert len(sample) == len(ACCENTS_TARGET)*CALLS_PER_ACCENT
assert sample.call_id.is_unique
assert all(sample.groupby("accent")["domain"].nunique() == CALLS_PER_ACCENT)
print("\n✓ Stage 2 completed. Save notebook + processed manifest + profile; move to 02_transcription.ipynb.")

### GitHub submission notes / reproducibility

- Commit this notebook, `selected_calls.csv`, `selected_calls.json`, `dataset_profile.json`, `provenance.json` (these include limited research metadata). Review attribution and sharing conditions for any derived content before publishing.
- **Do not** commit `data/raw_samples/` (large source WAV files). Use a `.gitignore` rule for audio and document a reproducible download method.
- Document that the dataset is role-played, not real-world customer data; it has no verified sentiment labels.
- Keep the source dataset's license, attribution, and evaluation-only intended use visible in the README.
- This step profiles the **diarization configuration**. Do not present its call-duration total as identical to the default configuration's speech-hour figure.
- **Next:** transcription with an open ASR baseline; speaker diarization on the merged audio; WER/DER measurements against reference labels; grounded summaries and customer sentiment.

*Source: [official dataset card](https://huggingface.co/datasets/apptek-com/apptek_callcenter_dialogues), [original paper](https://arxiv.org/abs/2604.27543).*